# Notebook setup

In [1]:
from pathlib import Path
import os

# show where the notebook is running
print("CWD before:", Path.cwd())

# Point to your package (adjust if needed)
# e.g. if your modules are under src/, add it to sys.path
import sys
sys.path.append(str(Path.cwd()))  # or Path("src").resolve()

print("CWD after:", Path.cwd())
# from data_processing.logging_utils import logger
# from data_processing.data_setup import create_data_directory

CWD before: /mnt/c/Users/Barrs/OneDrive - mail.tau.ac.il/MLHCproject/test2/mlhc_project/src
CWD after: /mnt/c/Users/Barrs/OneDrive - mail.tau.ac.il/MLHCproject/test2/mlhc_project/src


In [1]:
import pandas as pd
import pickle
from pathlib import Path
from typing import List, Tuple
import numpy as np

# from data_processing.integrated_data_preprocessor import IntegratedICUPreprocessor
# from cohort_data import get_cohort_hadm_ids_and_targets
# from logging_utils import logger
# from data_processing.data_setupd import create_data_directory

# Input CSV file paths
INITIAL_COHORT_CSV = "../csvs/initial_cohort.csv"    # Training/validation patient IDs
TEST_EXAMPLE_CSV = "../csvs/test_example.csv"        # Test set patient IDs

# Output directory for processed data
DATA_DIR = "data"

df_init = pd.read_csv(INITIAL_COHORT_CSV)
df_test = pd.read_csv(TEST_EXAMPLE_CSV)
# display(df_init.head()); display(df_test.head())
print("init shape:", df_init.shape, "test shape:", df_test.shape)

# For faster debug runs, sample a small subset (e.g., 100 patients)
INIT_SAMPLE_N = 200
TEST_SAMPLE_N = 50

# init_ids = df_init["subject_id"].astype(int).sample(min(INIT_SAMPLE_N, len(df_init)), random_state=42).tolist()
init_ids = df_init["subject_id"].astype(int).tolist()
test_ids = df_test["subject_id"].astype(int).sample(min(TEST_SAMPLE_N, len(df_test)), random_state=42).tolist()

print(len(init_ids), len(test_ids))

init shape: (32513, 1) test shape: (50, 1)
32513 50


## DB access sanity + cohort/targets only

In [2]:
import duckdb
from data_processing.data_extraction import DUCKDB_PATH  # or set your own path here

# db = Path("/mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb")
    # assert db.exists(), f"DB not found at {db}"
print("DUCKDB_PATH ->", DUCKDB_PATH)
# con = duckdb.connect(DUCKDB_PATH)
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# con = duckdb.connect(str(db), read_only=True)

# sanity checks
print(con.execute("PRAGMA database_list").fetchdf())
print(con.execute("SHOW TABLES").fetchdf())

DUCKDB_PATH -> /mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii.duckdb
   seq      name                                               file
0  570  mimiciii  /mnt/c/Users/Barrs/My Drive/MIMIC-III/mimiciii...
                  name
0           ADMISSIONS
1              CALLOUT
2           CAREGIVERS
3          CHARTEVENTS
4            CPTEVENTS
5       DATETIMEEVENTS
6        DIAGNOSES_ICD
7             DRGCODES
8                D_CPT
9      D_ICD_DIAGNOSES
10    D_ICD_PROCEDURES
11             D_ITEMS
12          D_LABITEMS
13            ICUSTAYS
14      INPUTEVENTS_CV
15      INPUTEVENTS_MV
16           LABEVENTS
17  MICROBIOLOGYEVENTS
18          NOTEEVENTS
19        OUTPUTEVENTS
20            PATIENTS
21       PRESCRIPTIONS
22  PROCEDUREEVENTS_MV
23      PROCEDURES_ICD
24            SERVICES
25           TRANSFERS


In [3]:
from data_processing.cohort_data import COHORT_SQL

# Register subject IDs as temporary table for SQL query
con.register("tmp_subject_ids", pd.DataFrame({"subject_id": init_ids}))

# Execute cohort SQL to get filtered admissions and target labels
df = con.execute(COHORT_SQL).fetchdf()
print(df.head())
print(df.shape)

# Extract admission IDs and target matrix
hadm_ids = df["hadm_id"].tolist()
targets = df[["mortality_event", "los_event", "readmission_event"]].reset_index(drop=True).values

   hadm_id  mortality_event  los_event  readmission_event
0   100003                0          0                  0
1   100006                0          1                  0
2   100007                0          1                  0
3   100009                0          0                  0
4   100010                0          0                  0
(22489, 4)


In [4]:
# Build a base table with everything we need
# returns all intermediate columns you need to check rules yourself (in Python):
#  subject_id, hadm_id, admittime, dischtime, deathtime, age, los_hours, has_chartevents_data, admission_rank, discharge_to_death_hours, discharge_to_readmission_hours, plus a computed helper died_within_54h.
#  It does not filter the cohort and does not output targets
base_sql = r"""
WITH ordered AS (
  SELECT
      a.subject_id::INTEGER            AS subject_id,
      a.hadm_id::INTEGER               AS hadm_id,
      a.admittime::TIMESTAMP           AS admittime,
      a.dischtime::TIMESTAMP           AS dischtime,
      a.deathtime::TIMESTAMP           AS deathtime,
      a.has_chartevents_data::INTEGER  AS has_chartevents_data,
      EXTRACT(year FROM AGE(a.admittime::TIMESTAMP, p.dob::TIMESTAMP))::INTEGER AS age,
      -- hospital LOS in hours
      EXTRACT(epoch FROM (a.dischtime::TIMESTAMP - a.admittime::TIMESTAMP)) / 3600.0 AS los_hours,
      -- death within first 54h of *admission* (Rule 5 exclusion)
      CASE
        WHEN a.deathtime IS NOT NULL
             AND EXTRACT(epoch FROM (a.deathtime::TIMESTAMP - a.admittime::TIMESTAMP)) / 3600.0 <= 54
        THEN 1 ELSE 0
      END AS died_within_54h,
      -- time to death after *discharge* (mortality target window)
      EXTRACT(epoch FROM (p.dod::TIMESTAMP - a.dischtime::TIMESTAMP)) / 3600.0 AS discharge_to_death_hours,
      -- time to next admission
      EXTRACT(epoch FROM (LEAD(a.admittime::TIMESTAMP) OVER (PARTITION BY a.subject_id ORDER BY a.admittime)
                          - a.dischtime::TIMESTAMP)) / 3600.0 AS discharge_to_readmission_hours,
      ROW_NUMBER() OVER (PARTITION BY a.subject_id ORDER BY a.admittime) AS admission_rank
  FROM admissions a
  JOIN patients  p ON a.subject_id = p.subject_id
  WHERE a.subject_id::INTEGER IN (SELECT subject_id FROM tmp_subject_ids)
)
SELECT *
FROM ordered
ORDER BY subject_id, admittime
"""
base_df = con.execute(base_sql).fetchdf()
base_df.head(10)
# print(base_df.shape)

,subject_id,hadm_id,admittime,dischtime,deathtime,has_chartevents_data,age,los_hours,died_within_54h,discharge_to_death_hours,discharge_to_readmission_hours,admission_rank
0,2,163353,2138-07-17 19:04:00,2138-07-21 15:48:00,NaT,1,0,92.733333,0,NaN,NaN,1
1,3,145834,2101-10-20 19:08:00,2101-10-31 13:58:00,NaT,1,76,258.833333,0,5410.033333,NaN,1
2,4,185777,2191-03-16 00:28:00,2191-03-23 18:41:00,NaT,1,47,186.216667,0,NaN,NaN,1
3,5,178980,2103-02-02 04:31:00,2103-02-04 12:15:00,NaT,1,0,55.733333,0,NaN,NaN,1
4,7,118037,2121-05-23 15:05:00,2121-05-27 11:57:00,NaT,1,0,92.866667,0,NaN,NaN,1
5,8,159514,2117-11-20 10:22:00,2117-11-24 14:20:00,NaT,1,0,99.966667,0,NaN,NaN,1
6,9,150750,2149-11-09 13:06:00,2149-11-14 10:15:00,2149-11-14 10:15:00,1,41,117.150000,0,-10.250000,NaN,1
7,11,194540,2178-04-16 06:18:00,2178-05-11 19:00:00,NaT,1,50,612.700000,0,4469.000000,NaN,1
8,12,112213,2104-08-07 10:15:00,2104-08-20 02:57:00,2104-08-20 02:57:00,1,72,304.700000,0,-2.950000,NaN,1
9,16,103251,2178-02-03 06:35:00,2178-02-05 10:51:00,NaT,1,0,52.266667,0,NaN,NaN,1


In [5]:
# Recreate the 5 rules in Python (ground truth inclusion)
MIN_AGE, MAX_AGE = 18, 89
MIN_LOS_HOURS = 54
MORTALITY_EVENT_HOURS = 720
LOS_EVENT_HOURS = 168
READMISSION_EVENT_HOURS = 720

def apply_rules(df):
    m1 = df["admission_rank"] == 1
    m2 = df["age"].between(MIN_AGE, MAX_AGE, inclusive="both")
    m3 = df["los_hours"] >= MIN_LOS_HOURS
    m4 = df["has_chartevents_data"] == 1
    m5 = df["died_within_54h"] == 0

    keep_mask = m1 & m2 & m3 & m4 & m5
    sizes = {
        "start": len(df),
        "rule1_first": int(m1.sum()),
        "rule2_age": int((m1 & m2).sum()),
        "rule3_los": int((m1 & m2 & m3).sum()),
        "rule4_chartevents": int((m1 & m2 & m3 & m4).sum()),
        "rule5_no_early_death": int(keep_mask.sum()),
    }
    # print(df[m3 & ~m5])
    return df[keep_mask].copy(), sizes

# print(base_df[base_df["died_within_54h"] == 1])
# print(base_df[base_df[base_df["los_hours"] >= MIN_LOS_HOURS]])
expected_cohort_df, sizes = apply_rules(base_df)
sizes, expected_cohort_df.shape


({'start': 41244,
  'rule1_first': 32513,
  'rule2_age': 25548,
  'rule3_los': 22927,
  'rule4_chartevents': 22493,
  'rule5_no_early_death': 22489},
 (22489, 12))

In [6]:
# Compute ground truth targets in Python
gt = expected_cohort_df.copy()

# Mortality: death within 30 days *after discharge* (using patients.dod vs dischtime)
gt["mortality_event"] = (gt["discharge_to_death_hours"] <= MORTALITY_EVENT_HOURS).astype(int)

# LOS>7 days target: NOTE this uses *hospital* LOS. If you intended ICU LOS, change source.
gt["los_event"] = (gt["los_hours"] > LOS_EVENT_HOURS).astype(int)

# Readmission within 30 days after discharge
gt["readmission_event"] = (gt["discharge_to_readmission_hours"] <= READMISSION_EVENT_HOURS).astype(int)

gt_targets = gt[["hadm_id","mortality_event","los_event","readmission_event"]].sort_values("hadm_id").reset_index(drop=True)
gt_targets.head()


,hadm_id,mortality_event,los_event,readmission_event
0,100003,0,0,0
1,100006,0,1,0
2,100007,0,1,0
3,100009,0,0,0
4,100010,0,0,0


In [7]:
from data_processing.cohort_data import get_cohort_hadm_ids_and_targets, COHORT_SQL
# Run COHORT_SQL and compare
sql_df = con.execute(COHORT_SQL).fetchdf()
sql_targets = sql_df[["hadm_id","mortality_event","los_event","readmission_event"]].sort_values("hadm_id").reset_index(drop=True)

# 1) Inclusion check: which HADM_IDs the SQL kept vs. Python rules
gt_hadm = set(gt_targets["hadm_id"].astype(int))
sql_hadm = set(sql_targets["hadm_id"].astype(int))

extra = sorted(sql_hadm - gt_hadm)    # in SQL but should be excluded
missing = sorted(gt_hadm - sql_hadm)  # expected by rules but not in SQL

print(f"GT size={len(gt_hadm)}  |  SQL size={len(sql_hadm)}")
print(f"Extra: {len(extra)}  Missing: {len(missing)}")
if extra:  display(base_df[base_df.hadm_id.isin(extra)].head(10))
if missing: display(base_df[base_df.hadm_id.isin(missing)].head(10))

# 2) Target equality for the intersection
common = sorted(gt_hadm & sql_hadm)
gt_common = gt_targets[gt_targets.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)
sql_common = sql_targets[sql_targets.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)

mismatch = (gt_common[["mortality_event","los_event","readmission_event"]].values !=
            sql_common[["mortality_event","los_event","readmission_event"]].values)
n_mismatch = int(mismatch.any(axis=1).sum())

print(f"Target mismatches on common HADM_IDs: {n_mismatch} / {len(common)}")
if n_mismatch:
    bad = gt_common.loc[mismatch.any(axis=1)]
    bad = bad.merge(sql_common, on="hadm_id", suffixes=("_gt","_sql"))
    display(bad.head(20))


GT size=22489  |  SQL size=22489
Extra: 0  Missing: 0
Target mismatches on common HADM_IDs: 0 / 22489


In [8]:
# 1) Get the ORIGINAL outputs (what your pipeline returns today)
hadm_ids_orig, targets_orig = get_cohort_hadm_ids_and_targets(con, init_ids)
print("cohort #hadm:", len(hadm_ids_orig))
print("targets shape:", targets_orig.shape)
pd.DataFrame(targets_orig, columns=["mortality","los>7d","readm≤30d"]).head()

# 2) Convert them to a tidy DataFrame (so we can compare apples-to-apples)
orig_df = (
    pd.DataFrame({
        "hadm_id": hadm_ids_orig,
        "mortality_event": targets_orig[:, 0].astype(int),
        "los_event": targets_orig[:, 1].astype(int),
        "readmission_event": targets_orig[:, 2].astype(int),
    })
    # .sort_values("hadm_id")
    .reset_index(drop=True)
)

# 4) Cohort inclusion diffs (who is in vs. who should be in)
orig_set = set(orig_df["hadm_id"].astype(int))
gt_set   = set(gt_targets["hadm_id"].astype(int))

extra   = sorted(orig_set - gt_set)   # present in ORIGINAL but not in GT (over-inclusion)
missing = sorted(gt_set - orig_set)   # present in GT but not in ORIGINAL (over-filtering)

print(f"Original size = {len(orig_set)} | GT size = {len(gt_set)}")
print(f"Extra (in ORIGINAL, not GT): {len(extra)}")
print(f"Missing (in GT, not ORIGINAL): {len(missing)}")

if extra:
    display(base_df[base_df.hadm_id.isin(extra)][
        ["subject_id","hadm_id","admittime","dischtime","deathtime","age","los_hours","has_chartevents_data","admission_rank"]
    ].head(10))

if missing:
    display(base_df[base_df.hadm_id.isin(missing)][
        ["subject_id","hadm_id","admittime","dischtime","deathtime","age","los_hours","has_chartevents_data","admission_rank"]
    ].head(10))

# 5) Target label comparison on the intersection
common = sorted(orig_set & gt_set)
orig_common = orig_df[orig_df.hadm_id.isin(common)].reset_index(drop=True)
gt_common   = gt_targets[gt_targets.hadm_id.isin(common)].reset_index(drop=True)

# Vectorized mismatch check
cols = ["mortality_event","los_event","readmission_event"]
mismatch_mask = (orig_common[cols].values != gt_common[cols].values)
rows_with_any_mismatch = mismatch_mask.any(axis=1)
n_rows_bad = int(rows_with_any_mismatch.sum())

print(f"Target mismatches on common HADM_IDs: {n_rows_bad} / {len(common)}")
if n_rows_bad:
    bad = (
        orig_common.loc[rows_with_any_mismatch, ["hadm_id"] + cols]
        .merge(gt_common.loc[rows_with_any_mismatch, ["hadm_id"] + cols],
               on="hadm_id", suffixes=("_orig", "_gt"))
    )
    display(bad.head(20))

19:11:36.360 Started get_cohort_hadm_ids_and_targets
19:11:36.637 Finished get_cohort_hadm_ids_and_targets
cohort #hadm: 22489
targets shape: (22489, 3)
Original size = 22489 | GT size = 22489
Extra (in ORIGINAL, not GT): 0
Missing (in GT, not ORIGINAL): 0
Target mismatches on common HADM_IDs: 0 / 22489


In [15]:
# label distribution check (stratification sanity)
lab = pd.DataFrame(targets, columns=["mortality","los","readm"])
lab["combo"] = (lab["mortality"].astype(int).astype(str) +
                lab["los"].astype(int).astype(str) +
                lab["readm"].astype(int).astype(str))
lab["combo"].value_counts(normalize=True).rename("freq").to_frame()

,freq
combo,
010,0.443194
000,0.391436
110,0.074481
100,0.043443
011,0.026991
001,0.013829
111,0.004447
101,0.002179


## Static features only

In [9]:
from data_processing.static_data import (
    STATIC_SQL,                      # the query under test
    CATEGORICAL_COLUMNS, NUMERIC_COLUMNS, NUMERIC_COLUMNS_WITH_MISSING,
    BINARY_COLUMNS, STATIC_COLUMNS,
    HEIGHT_IN_ITEMIDS, HEIGHT_CM_ITEMIDS, WEIGHT_KG_ITEMIDS, WEIGHT_LB_ITEMIDS, WEIGHT_OZ_ITEMIDS,
    VASOPRESSOR_CV_ITEMIDS, VASOPRESSOR_MV_ITEMIDS,
    VENTILATION_PROCEDURE_ITEMIDS, VENTILATION_CHART_ITEMIDS,
    RRT_PROCEDURE_ITEMIDS, RRT_CHART_ITEMIDS,
    SEDATION_CV_ITEMIDS, SEDATION_MV_ITEMIDS,
    WINDOW_HOURS, IN_TO_CM_FACTOR, LB_TO_KG_FACTOR, ANTIBIOTIC_REGEX
)
from data_processing.static_data import get_static_data  # original function

base_hadm_ids = gt["hadm_id"].tolist()
base_targets = gt_targets[["mortality_event","los_event","readmission_event"]]
print("base cohort #hadm:", len(base_hadm_ids))
print("base targets shape:", base_targets.shape)
print("orig cohort #hadm:", len(hadm_ids_orig))
print("orig targets shape:", targets_orig.shape)

base cohort #hadm: 22489
base targets shape: (22489, 3)
orig cohort #hadm: 22489
orig targets shape: (22489, 3)


In [4]:
from data_processing.static_data import get_static_data, STATIC_COLUMNS

static_raw = get_static_data(con, hadm_ids)
print("static_raw shape:", static_raw.shape)

# peek as DataFrame with original column ordering
# from yourpkg.static_data import STATIC_COLUMNS
pd.DataFrame(static_raw, columns=STATIC_COLUMNS).head(10)

18:19:48.279 Started get_static_data
18:20:23.692 Finished get_static_data
static_raw shape: (22489, 18)


,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,height,weight,hours_to_first_icu,received_vasopressor,recieved_mechanical_ventilation,received_rrt,received_sedation,received_antibiotic,reached_icu
0,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,NaN,85.3,0,1,1,0,0,1,1
1,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,NaN,NaN,0,0,1,0,0,1,1
2,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,NaN,NaN,5,0,1,0,0,1,1
3,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,182.88,115.0,19,0,1,0,1,1,1
4,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,NaN,71.0,14,1,1,0,0,0,1
5,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,NaN,78.9,47,0,0,0,0,1,1
6,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,137.16,49.7,0,0,1,0,1,1,1
7,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,NaN,NaN,<NA>,0,0,0,0,1,0
8,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,175.26,90.7,2,1,1,0,1,1,1
9,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,NaN,62.9,0,0,1,0,0,0,1


In [12]:
# --- Modular recomputation (from your cohort.py/queries.py) ---
ICU_INTIME1 = f"""
    WITH first_icu AS (
            SELECT
                i.hadm_id,
                MIN(i.intime)::TIMESTAMP AS first_icu_intime
            FROM icustays i
            JOIN admissions a ON i.hadm_id = a.hadm_id
            WHERE i.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids)
              AND i.intime::TIMESTAMP BETWEEN a.admittime::TIMESTAMP 
                                          AND a.admittime::TIMESTAMP + INTERVAL {WINDOW_HOURS} HOURS
            GROUP BY i.hadm_id
        )
        SELECT
            a.hadm_id::INTEGER AS hadm_id,
            fi.first_icu_intime,
            datediff('hour', a.admittime::TIMESTAMP, fi.first_icu_intime) AS hours_to_first_icu,             -- Time (in hours) from hospital admission to first ICU intime
        FROM admissions a
        LEFT JOIN first_icu fi ON a.hadm_id = fi.hadm_id
        WHERE a.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids)
    """
import numpy as np
from cohort_constractions import (
    add_first_icu_intime, add_first_height, add_first_weight,
    add_received_vasopressor_flag, add_received_sedation_flag,
    add_was_mechanically_ventilated_flag, add_received_rrt_flag,
    add_received_antibiotic_flag
)
# def add_first_icu_intime(con, hadm_ids: List[int], cohort_df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Add earliest ICU intime within WINDOW_HOURS (48h) of admission for each hadm_id.
#     Returns:
#         cohort_df with an added column: first_icu_intime (nullable).
#     """
#     con.register("tmp_hadm_ids", pd.DataFrame({"hadm_id": hadm_ids}))
#     icu = con.execute(ICU_INTIME1).fetchdf()
#     display(icu.head())
#     cohort_df = cohort_df.merge(icu, on="hadm_id", how="left")
#     return cohort_df

def static_via_steps(con, hadm_ids):
    # Build the SAME base fields as STATIC_SQL expects (using DuckDB so age is identical)
    con.register("tmp_hadm_ids", pd.DataFrame({"hadm_id": hadm_ids}))
    base = con.execute("""
        SELECT 
            a.hadm_id::INTEGER AS hadm_id,
            a.admission_type,
            a.admission_location,
            a.insurance,
            a.language,
            a.religion,
            a.marital_status,
            a.ethnicity,
            CASE WHEN p.gender = 'M' THEN 1 ELSE 0 END AS gender,
            EXTRACT(year FROM AGE(a.admittime::TIMESTAMP, p.dob::TIMESTAMP))::INTEGER AS age,
            a.admittime::TIMESTAMP AS admittime
        FROM admissions a
        JOIN patients p ON a.subject_id = p.subject_id
        WHERE a.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids)
        ORDER BY a.hadm_id
    """).fetchdf()

    # Standardize column names to lowercase
    base.columns = base.columns.str.lower()

    # Fill missing categoricals same as the production extractor
    for col in ["admission_type","admission_location","insurance","language","religion","marital_status","ethnicity"]:
        base[col] = base[col].fillna("missing")

    # Add ICU/time-window features using your modular helpers
    coh = base.copy()
    coh = add_first_height(con, hadm_ids, coh)                     # height_cm
    coh = add_first_weight(con, hadm_ids, coh)                     # weight_kg
    coh = add_first_icu_intime(con, hadm_ids, coh)                 # first_icu_intime
    # coh = add_received_vasopressor_flag(con, hadm_ids, coh)        # received_vasopressor
    # coh = add_received_sedation_flag(con, hadm_ids, coh)           # received_sedation
    # coh = add_was_mechanically_ventilated_flag(con, hadm_ids, coh) # was_mechanically_ventilated
    # coh = add_received_rrt_flag(con, hadm_ids, coh)                # received_rrt
    # coh = add_received_antibiotic_flag(con, hadm_ids, coh)         # received_antibiotic

    # Adapt names/units so they MATCH STATIC_COLUMNS
    coh["height"] = coh["height_cm"]
    coh["weight"] = coh["weight_kg"]
    coh["reached_icu"] = coh["first_icu_intime"].notna().astype(int)
    # # match the (typo) column name in STATIC_COLUMNS
    # coh["recieved_mechanical_ventilation"] = coh["was_mechanically_ventilated"].astype(int)

    # # Keep exactly the STATIC schema
    # out = coh[["hadm_id"] + STATIC_COLUMNS].copy()
    # # Ensure numeric dtypes consistent
    # out["age"] = out["age"].astype(int)
    # for b in ["received_vasopressor","recieved_mechanical_ventilation","received_rrt","received_sedation","received_antibiotic","reached_icu"]:
    #     out[b] = out[b].fillna(0).astype(int)
    out = coh.copy()
    # print(out.head(20))
    return out

df_step = static_via_steps(con, base_hadm_ids)
display(df_step.head(10))

,hadm_id,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,admittime,height_cm,weight_kg,first_icu_intime,hours_to_first_icu,height,weight,reached_icu
0,100003,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,2150-04-17 15:34:00,NaN,85.139288,2150-04-17 15:35:42,0,NaN,85.139288,1
1,100006,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,2108-04-06 15:49:00,NaN,NaN,2108-04-06 15:50:15,0,NaN,NaN,1
2,100007,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,2145-03-31 05:33:00,NaN,NaN,2145-03-31 10:17:23,5,NaN,NaN,1
3,100009,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,2162-05-16 15:56:00,182.88,115.000000,2162-05-17 10:18:31,19,182.88,115.000000,1
4,100010,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,2109-12-10 07:15:00,NaN,71.000000,2109-12-10 21:58:01,14,NaN,71.000000,1
5,100012,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,2177-03-12 11:48:00,NaN,78.900000,2177-03-14 10:52:23,47,NaN,78.900000,1
6,100016,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,2188-05-24 13:06:00,137.16,49.700000,2188-05-24 13:07:20,0,137.16,49.700000,1
7,100021,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,2109-08-17 10:55:00,NaN,NaN,NaT,<NA>,NaN,NaN,0
8,100024,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,2170-09-19 07:30:00,175.26,90.700000,2170-09-19 09:44:32,2,175.26,90.700000,1
9,100028,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,2142-12-23 18:06:00,NaN,62.777184,2142-12-23 18:07:12,0,NaN,62.777184,1


In [ ]:
def static_via_sql(con, hadm_ids):
    # Register all temp tables exactly as the function does
    con.register("tmp_hadm_ids", pd.DataFrame({"hadm_id": hadm_ids}))
    con.register("tmp_height_in_itemids", pd.DataFrame({"itemid": HEIGHT_IN_ITEMIDS}))
    con.register("tmp_height_cm_itemids", pd.DataFrame({"itemid": HEIGHT_CM_ITEMIDS}))
    con.register("tmp_weight_kg_itemids", pd.DataFrame({"itemid": WEIGHT_KG_ITEMIDS}))
    con.register("tmp_weight_lb_itemids", pd.DataFrame({"itemid": WEIGHT_LB_ITEMIDS}))
    con.register("tmp_weight_oz_itemids", pd.DataFrame({"itemid": WEIGHT_OZ_ITEMIDS}))
    con.register("tmp_vaso_cv_itemids", pd.DataFrame({"itemid": VASOPRESSOR_CV_ITEMIDS}))
    con.register("tmp_vaso_mv_itemids", pd.DataFrame({"itemid": VASOPRESSOR_MV_ITEMIDS}))
    con.register("tmp_vent_proc_itemids", pd.DataFrame({"itemid": VENTILATION_PROCEDURE_ITEMIDS}))
    con.register("tmp_vent_chart_itemids", pd.DataFrame({"itemid": VENTILATION_CHART_ITEMIDS}))
    con.register("tmp_rrt_proc_itemids", pd.DataFrame({"itemid": RRT_PROCEDURE_ITEMIDS}))
    con.register("tmp_rrt_chart_itemids", pd.DataFrame({"itemid": RRT_CHART_ITEMIDS}))
    con.register("tmp_sed_cv_itemids", pd.DataFrame({"itemid": SEDATION_CV_ITEMIDS}))
    con.register("tmp_sed_mv_itemids", pd.DataFrame({"itemid": SEDATION_MV_ITEMIDS}))
    
    df = con.execute(STATIC_SQL).fetchdf()
    df.columns = df.columns.str.lower()
    # align categorical NA handling to the production function
    for col in CATEGORICAL_COLUMNS:
        df[col] = df[col].fillna("missing")
    # out = df[["hadm_id"] + STATIC_COLUMNS].copy()
    out = df.copy()
    return out

df_sql = static_via_sql(con, base_hadm_ids)
display(df_sql.head(10))

In [ ]:
def compare_static_frames(df_sql, df_steps, atol=1e-9, max_show=15):
    # align rows
    left = df_sql.sort_values("hadm_id").reset_index(drop=True)
    right = df_steps.sort_values("hadm_id").reset_index(drop=True)

    # check cohort membership first
    left_ids = set(left.hadm_id.astype(int))
    right_ids = set(right.hadm_id.astype(int))
    extra = sorted(left_ids - right_ids)
    missing = sorted(right_ids - left_ids)
    print(f"#rows SQL={len(left)}  STEPS={len(right)}  |  extra={len(extra)}  missing={len(missing)}")
    if extra:
        display(left[left.hadm_id.isin(extra)].head(max_show))
    if missing:
        display(right[right.hadm_id.isin(missing)].head(max_show))

    # intersect for value comparison
    common = sorted(left_ids & right_ids)
    L = left[left.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)
    R = right[right.hadm_id.isin(common)].sort_values("hadm_id").reset_index(drop=True)

    cat_cols = CATEGORICAL_COLUMNS
    num_cols = NUMERIC_COLUMNS + BINARY_COLUMNS

    # categorical exact compare
    cat_bad = []
    for c in cat_cols:
        ne = (L[c].astype(str).fillna("missing") != R[c].astype(str).fillna("missing"))
        if ne.any():
            cat_bad.append(c)
            print(f"[CAT] mismatch in '{c}': {int(ne.sum())}/{len(ne)} rows")
            display(pd.concat([L.loc[ne, ["hadm_id", c]].rename(columns={c: f"{c}_sql"}),
                               R.loc[ne, [c]].rename(columns={c: f"{c}_steps"})], axis=1).head(max_show))

    # numeric close compare
    num_bad = []
    for c in num_cols:
        lv = pd.to_numeric(L[c]).astype(float)
        rv = pd.to_numeric(R[c]).astype(float)
        same = np.isclose(lv.values, rv.values, equal_nan=True, atol=atol)
        if ~same.all():
            num_bad.append(c)
            ne = ~same
            print(f"[NUM] mismatch in '{c}': {int(ne.sum())}/{len(ne)} rows")
            display(pd.DataFrame({
                "hadm_id": L.loc[ne, "hadm_id"],
                f"{c}_sql": lv.loc[ne].values,
                f"{c}_steps": rv.loc[ne].values
            }).head(max_show))

    if not extra and not missing and not cat_bad and not num_bad:
        print("✅ Static features match perfectly (schema & values).")
    else:
        print("⚠️ Differences detected. See tables above.")

# hadm_ids you want to validate (e.g., from your cohort check)
df_sql  = static_via_sql(con, base_hadm_ids)
df_step = static_via_steps(con, base_hadm_ids)

compare_static_frames(df_sql, df_step)


In [40]:
height_ids = [920, 1394, 4187, 3486, 226730, 3485, 4188, 226707]
weight_ids = [762, 763, 3723, 3580, 226512, 224639, 3581, 226531, 3582]

q_height_present = f"""
SELECT di.itemid, di.label, di.linksto, di.dbsource, COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
WHERE ce.itemid IN ({",".join(map(str, height_ids))})
GROUP BY 1,2,3,4
ORDER BY n DESC;
"""

q_weight_present = f"""
SELECT di.itemid, di.label, di.linksto, di.dbsource, COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
WHERE ce.itemid IN ({",".join(map(str, weight_ids))})
GROUP BY 1,2,3,4
ORDER BY n DESC;
"""

height_present = con.execute(q_height_present).fetchdf()
weight_present = con.execute(q_weight_present).fetchdf()

display(height_present)
display(weight_present)


,ITEMID,LABEL,LINKSTO,DBSOURCE,n
0,4188,Length in cm,chartevents,carevue,148996
1,4187,Length Calc Inches,chartevents,carevue,148995
2,920,Admit Ht,chartevents,carevue,41704
3,226707,Height,chartevents,metavision,12015
4,226730,Height (cm),chartevents,metavision,12015
5,3486,Length in Inches,chartevents,carevue,407
6,3485,Length Calc (cm),chartevents,carevue,388
7,1394,Height Inches,chartevents,carevue,26


,ITEMID,LABEL,LINKSTO,DBSOURCE,n
0,3723,Birth Weight (kg),chartevents,carevue,435385
1,3580,Present Weight (kg),chartevents,carevue,433003
2,3581,Present Weight (lb),chartevents,carevue,432997
3,3582,Present Weight (oz),chartevents,carevue,432996
4,763,Daily Weight,chartevents,carevue,47689
5,224639,Daily Weight,chartevents,metavision,46452
6,226531,Admission Weight (lbs.),chartevents,metavision,46255
7,762,Admit Wt,chartevents,carevue,41704
8,226512,Admission Weight (Kg),chartevents,metavision,22604


In [54]:
HEIGHT_IN_ITEMIDS = [4187, 3486, 1394, 920, 226707]      # inches
HEIGHT_CM_ITEMIDS = [4188, 3485, 226730]                 # centimeters

WEIGHT_KG_ITEMIDS = [762, 763, 3580, 226512, 224639]     # kilograms
WEIGHT_LB_ITEMIDS = [3581, 226531]                       # pounds
WEIGHT_OZ_ITEMIDS = [3582]

# Heights (48h, cohort only)
q_h = f"""
SELECT
  ce.itemid, di.label,
  MIN(CASE WHEN ce.itemid IN ({",".join(map(str, HEIGHT_IN_ITEMIDS))})
           THEN CAST(ce.valuenum AS DOUBLE) * 2.54
           ELSE CAST(ce.valuenum AS DOUBLE)
      END) AS min_v,
  MEDIAN(CASE WHEN ce.itemid IN ({",".join(map(str, HEIGHT_IN_ITEMIDS))})
              THEN CAST(ce.valuenum AS DOUBLE) * 2.54
              ELSE CAST(ce.valuenum AS DOUBLE)
         END) AS p50,
  MAX(CASE WHEN ce.itemid IN ({",".join(map(str, HEIGHT_IN_ITEMIDS))})
           THEN CAST(ce.valuenum AS DOUBLE) * 2.54
           ELSE CAST(ce.valuenum AS DOUBLE)
      END) AS max_v,
  COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
JOIN admissions a ON ce.hadm_id = a.hadm_id
WHERE ce.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids)
  AND ce.itemid IN ({",".join(map(str, HEIGHT_IN_ITEMIDS + HEIGHT_CM_ITEMIDS))})
  AND ce.valuenum IS NOT NULL
  AND ce.error::INTEGER = 0FROM chartevents ce
  AND ce.charttime::TIMESTAMP BETWEEN a.admittime::TIMESTAMP AND a.admittime::TIMESTAMP + INTERVAL 48 HOURS
GROUP BY ce.itemid, di.label
ORDER BY n DESC;
"""
display(con.execute(q_h).fetchdf())

# Weights (48h, cohort only)
q_w = f"""
SELECT ce.itemid, di.label,
       MIN(CAST(ce.valuenum AS DOUBLE))       AS min_v,
       MEDIAN(CAST(ce.valuenum AS DOUBLE))    AS p50,
       MAX(CAST(ce.valuenum AS DOUBLE))       AS max_v,
       COUNT(*)                                AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
JOIN admissions a ON ce.hadm_id = a.hadm_id
-- Ensure that all weights are in kg and heights are in centimeters
CASE
  WHEN c.itemid   IN (3581, 226531)
    THEN c.valuenum * 0.45359237
  WHEN c.itemid   IN (3582)
    THEN c.valuenum * 0.0283495231
  ELSE c.valuenum
END AS valuenum
WHERE ce.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids)
  AND ce.itemid IN ({",".join(map(str, WEIGHT_KG_ITEMIDS + WEIGHT_LB_ITEMIDS))})
  AND ce.valuenum IS NOT NULL
  AND ce.error::INTEGER = 0
  AND ce.charttime::TIMESTAMP BETWEEN a.admittime::TIMESTAMP AND a.admittime::TIMESTAMP + INTERVAL 48 HOURS
GROUP BY ce.itemid, di.label
ORDER BY n DESC;
"""
display(con.execute(q_w).fetchdf())


ParserException: Parser Error: syntax error at or near "FROM"

In [ ]:
# If you have your cohort hadm_ids handy:
con.register("tmp_hadm_ids_verify", pd.DataFrame({"hadm_id": base_hadm_ids}))

q_height_ranges_48h = """
SELECT ce.itemid, di.label,
       MIN(ce.valuenum) AS min_v,
       MEDIAN(ce.valuenum) AS p50,
       MAX(ce.valuenum) AS max_v,
       COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
JOIN admissions a ON ce.hadm_id = a.hadm_id
WHERE ce.itemid IN ({ids})
  AND ce.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids_verify)
  AND ce.valuenum IS NOT NULL
  AND ce.error::INTEGER = 0
  AND ce.charttime::TIMESTAMP BETWEEN a.admittime::TIMESTAMP AND a.admittime::TIMESTAMP + INTERVAL 48 HOURS
GROUP BY ce.itemid, di.label
ORDER BY n DESC;
""".format(ids=",".join(map(str, height_ids)))

q_weight_ranges_48h = """
SELECT ce.itemid, di.label,
       MIN(ce.valuenum) AS min_v,
       MEDIAN(ce.valuenum) AS p50,
       MAX(ce.valuenum) AS max_v,
       COUNT(*) AS n
FROM chartevents ce
JOIN d_items di USING (itemid)
JOIN admissions a ON ce.hadm_id = a.hadm_id
WHERE ce.itemid IN ({ids})
  AND ce.hadm_id::INTEGER IN (SELECT hadm_id FROM tmp_hadm_ids_verify)
  AND ce.valuenum IS NOT NULL
  AND ce.error::INTEGER = 0
  AND ce.charttime::TIMESTAMP BETWEEN a.admittime::TIMESTAMP AND a.admittime::TIMESTAMP + INTERVAL 48 HOURS
GROUP BY ce.itemid, di.label
ORDER BY n DESC;
""".format(ids=",".join(map(str, weight_ids)))

height_ranges_48h = con.execute(q_height_ranges_48h).fetchdf()
weight_ranges_48h = con.execute(q_weight_ranges_48h).fetchdf()

display(height_ranges_48h)
display(weight_ranges_48h)


,ITEMID,LABEL,min_v,p50,max_v,n
0,226730,Height (cm),0,170,79,3481
1,226707,Height,0,67,98,3479


,ITEMID,LABEL,min_v,p50,max_v,n
0,226531,Admission Weight (lbs.),0,176,99.9,11967
1,226512,Admission Weight (Kg),1,70,99.9,7019
2,224639,Daily Weight,.5,69.8,99.9,5181


,hadm_id,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,admittime,height_cm,weight_kg,first_icu_intime,height,weight,reached_icu
0,100003,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,2150-04-17 15:34:00,NaN,85.139288,2150-04-17 15:35:42,NaN,85.139288,1
1,100006,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,2108-04-06 15:49:00,NaN,NaN,2108-04-06 15:50:15,NaN,NaN,1
2,100007,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,2145-03-31 05:33:00,NaN,NaN,2145-03-31 10:17:23,NaN,NaN,1
3,100009,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,2162-05-16 15:56:00,182.88,115.000000,2162-05-17 10:18:31,182.88,115.000000,1
4,100010,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,2109-12-10 07:15:00,NaN,71.000000,2109-12-10 21:58:01,NaN,71.000000,1
5,100012,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,2177-03-12 11:48:00,NaN,78.900000,2177-03-14 10:52:23,NaN,78.900000,1
6,100016,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,2188-05-24 13:06:00,137.16,49.700000,2188-05-24 13:07:20,137.16,49.700000,1
7,100021,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,2109-08-17 10:55:00,NaN,NaN,NaT,NaN,NaN,0
8,100024,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,2170-09-19 07:30:00,175.26,90.700000,2170-09-19 09:44:32,175.26,90.700000,1
9,100028,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,2142-12-23 18:06:00,NaN,62.777184,2142-12-23 18:07:12,NaN,62.777184,1


16:30:33.865 Started get_static_data
16:31:10.835 Finished get_static_data
static_raw shape: (22489, 17)


,admission_type,admission_location,insurance,language,religion,marital_status,ethnicity,gender,age,height,weight,received_vasopressor,recieved_mechanical_ventilation,received_rrt,received_sedation,received_antibiotic,reached_icu
0,EMERGENCY,EMERGENCY ROOM ADMIT,Private,ENGL,NOT SPECIFIED,SINGLE,WHITE,1,59,NaN,85.3,1,1,0,0,1,1
1,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,NOT SPECIFIED,SINGLE,BLACK/AFRICAN AMERICAN,0,48,NaN,NaN,0,1,0,0,1,1
2,EMERGENCY,EMERGENCY ROOM ADMIT,Private,missing,JEWISH,MARRIED,WHITE,0,73,NaN,NaN,0,1,0,0,1,1
3,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Private,missing,CATHOLIC,MARRIED,WHITE,1,60,182.88,115.0,0,1,0,1,1,1
4,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Private,ENGL,EPISCOPALIAN,MARRIED,WHITE,0,54,NaN,71.0,1,1,0,0,0,1
5,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,1,67,NaN,78.9,0,0,0,0,1,1
6,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,PROTESTANT QUAKER,SINGLE,WHITE,1,55,137.16,49.7,0,1,0,1,1,1
7,EMERGENCY,EMERGENCY ROOM ADMIT,Medicaid,SPAN,UNOBTAINABLE,MARRIED,HISPANIC OR LATINO,1,54,NaN,NaN,0,0,0,0,1,0
8,ELECTIVE,PHYS REFERRAL/NORMAL DELI,Medicare,ENGL,NOT SPECIFIED,MARRIED,UNKNOWN/NOT SPECIFIED,1,71,175.26,90.7,1,1,0,1,1,1
9,EMERGENCY,CLINIC REFERRAL/PREMATURE,Medicare,ENGL,CATHOLIC,SINGLE,WHITE,0,72,NaN,62.9,0,1,0,0,0,1


## Time-series features only

In [ ]:
from data_processing.timeseries_data import get_timeseries_data, TIMESERIES_COLUMNS

timeseries_data, timeseries_missingness = get_timeseries_data(con, hadm_ids)
# con.close()

print("ts_data:", timeseries_data.shape, "ts_miss:", timeseries_missingness.shape)
# tiny peek
N, H, F = timeseries_data.shape
timeseries_data[0, :3, :8]  # first patient, first 3 hours, first 8 features
pd.DataFrame(timeseries_data, columns=TIMESERIES_COLUMNS).head()

16:12:31.771 Started get_timeseries_data
16:12:40.256 Finished get_timeseries_data
ts_data: (61, 48, 216) ts_miss: (61, 48, 216)


ValueError: Must pass 2-d input. shape=(61, 48, 216)